# 02 - Model Training and Selection

This notebook reproduces the validation-driven experiment journey. It starts with simple baselines, compares classical NLP models, and later records controlled tuning and ablation experiments. The test set is intentionally not evaluated here.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
from datetime import datetime

import joblib
import pandas as pd
from datasets import load_dataset
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import classification_report_dataframe, evaluate_predictions
from src.paths import CHECKPOINTS_DIR, MODELS_DIR, PROCESSED_DATA_DIR, REPORTS_DIR
from src.preprocessing import preprocess_text

pd.set_option("display.max_colwidth", 140)

## Load Prepared Splits

The notebook prefers local prepared parquet files from `01_data_setup.ipynb`. If they are absent, it falls back to the Hugging Face dataset and recreates the minimal text column locally in memory.

In [ ]:
DATASET_NAME = "QCRI/HumAID-all"


def load_split(split: str) -> pd.DataFrame:
    local_path = PROCESSED_DATA_DIR / f"humaid_{split}_minimal.parquet"
    if local_path.exists():
        return pd.read_parquet(local_path)

    dataset = load_dataset(DATASET_NAME, split=split)
    frame = dataset.to_pandas()
    frame["text_minimal"] = frame["tweet_text"].apply(preprocess_text)
    return frame


train_df = load_split("train")
validation_df = load_split("validation")

X_train = train_df["text_minimal"]
y_train = train_df["class_label"]
X_val = validation_df["text_minimal"]
y_val = validation_df["class_label"]
class_names = sorted(y_train.unique())

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Labels:", len(class_names))

## Evaluation Helper

Macro-F1 is the primary selection metric because the dataset is imbalanced and minority operational classes such as missing people and urgent needs should influence model choice.

In [ ]:
experiment_results: list[dict] = []
trained_models: dict[str, object] = {}


def record_result(experiment_id: str, display_name: str, predictions) -> dict:
    result = evaluate_predictions(
        experiment_name=experiment_id,
        y_true=y_val,
        y_pred=predictions,
        labels=class_names,
    )
    result["name"] = display_name
    experiment_results.append(result)
    return result


def fit_pipeline(experiment_id: str, display_name: str, pipeline: Pipeline, X_source=X_train) -> dict:
    pipeline.fit(X_source, y_train)
    predictions = pipeline.predict(X_val)
    trained_models[experiment_id] = pipeline
    return record_result(experiment_id, display_name, predictions)


def show_results() -> pd.DataFrame:
    return (
        pd.DataFrame(experiment_results)
        .sort_values(["macro_f1", "weighted_f1", "accuracy"], ascending=False)
        .reset_index(drop=True)
    )

## E0 - Most-Frequent Dummy Baseline

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train.to_frame(), y_train)
dummy_predictions = dummy_model.predict(X_val.to_frame())
record_result("E0_dummy_most_frequent", "Most-frequent dummy baseline", dummy_predictions)

## E1 - CountVectorizer Unigram + MultinomialNB

In [ ]:
e1_pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False)),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E1_count_unigram_nb", "Count unigram + MultinomialNB", e1_pipeline)

## E2 - CountVectorizer Unigram/Bigram + MultinomialNB

In [ ]:
e2_pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E2_count_1_2gram_nb", "Count unigram/bigram + MultinomialNB", e2_pipeline)

## E3 - TF-IDF Unigram + MultinomialNB

In [ ]:
e3_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", MultinomialNB(alpha=1.0)),
    ]
)
fit_pipeline("E3_tfidf_unigram_nb", "TF-IDF unigram + MultinomialNB", e3_pipeline)

## E4 - TF-IDF Unigram + Logistic Regression

In [ ]:
e4_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight=None, random_state=42),
        ),
    ]
)
fit_pipeline("E4_tfidf_unigram_lr", "TF-IDF unigram + Logistic Regression", e4_pipeline)

## E5 - Balanced Logistic Regression

In [ ]:
e5_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 1), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42),
        ),
    ]
)
fit_pipeline("E5_tfidf_unigram_lr_balanced", "TF-IDF unigram + balanced Logistic Regression", e5_pipeline)

## E6 - Word Unigram/Bigram Balanced Logistic Regression

In [ ]:
e6_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        (
            "classifier",
            LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42),
        ),
    ]
)
fit_pipeline("E6_tfidf_1_2gram_lr_balanced", "TF-IDF unigram/bigram + balanced Logistic Regression", e6_pipeline)

## E7 - LinearSVC

In [ ]:
e7_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E7_tfidf_1_2gram_linearsvc_balanced", "TF-IDF unigram/bigram + balanced LinearSVC", e7_pipeline)

## E8 - Character TF-IDF LinearSVC

In [ ]:
e8_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, lowercase=False, sublinear_tf=True),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E8_char_3_5gram_linearsvc_balanced", "Character TF-IDF + balanced LinearSVC", e8_pipeline)

## E9 - Word + Character Feature Union

In [ ]:
e9_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)),
                    ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, lowercase=False)),
                ]
            ),
        ),
        ("classifier", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)),
    ]
)
fit_pipeline("E9_word_char_tfidf_linearsvc_balanced", "Word + character TF-IDF + balanced LinearSVC", e9_pipeline)

## Baseline Comparison Through E9

Balanced Logistic Regression with word unigrams and bigrams is the strongest model family at this stage. Later cells tune `C` and test controlled raw/preprocessing ablations without touching the test set.

In [ ]:
results_df = show_results()
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(REPORTS_DIR / "validation_results.csv", index=False)
results_df

## E10 - Logistic Regression C Tuning

E6 established the strongest model family. This section tunes the regularization strength on validation Macro-F1 while tracking the two critical recalls.

In [ ]:
C_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0]
c_experiment_results: list[dict] = []
c_trained_models: dict[float, Pipeline] = {}

for c_value in C_VALUES:
    pipeline = Pipeline(
        [
            (
                "vectorizer",
                TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=c_value,
                    max_iter=1000,
                    solver="liblinear",
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)
    report_df = classification_report_dataframe(y_val, predictions, labels=class_names)

    c_experiment_results.append(
        {
            "C": c_value,
            "accuracy": float(report_df.loc["accuracy", "precision"]),
            "macro_f1": float(report_df.loc["macro avg", "f1-score"]),
            "weighted_f1": float(report_df.loc["weighted avg", "f1-score"]),
            "missing_precision": float(report_df.loc["missing_or_found_people", "precision"]),
            "missing_recall": float(report_df.loc["missing_or_found_people", "recall"]),
            "urgent_precision": float(report_df.loc["requests_or_urgent_needs", "precision"]),
            "urgent_recall": float(report_df.loc["requests_or_urgent_needs", "recall"]),
        }
    )
    c_trained_models[c_value] = pipeline

c_results_df = (
    pd.DataFrame(c_experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
best_c = float(c_results_df.loc[0, "C"])
best_c_pipeline = c_trained_models[best_c]
best_c_predictions = best_c_pipeline.predict(X_val)
e10_result = evaluate_predictions(
    "E10_tfidf_1_2gram_lr_balanced_C2",
    y_val,
    best_c_predictions,
    labels=class_names,
)
e10_result["name"] = "Tuned TF-IDF unigram/bigram + balanced Logistic Regression"

c_results_df

In [ ]:
def upsert_experiment(result: dict) -> pd.DataFrame:
    global experiment_results
    experiment_results = [
        existing for existing in experiment_results if existing["experiment"] != result["experiment"]
    ]
    experiment_results.append(result)
    return show_results()


results_df = upsert_experiment(e10_result)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
c_results_df.to_csv(REPORTS_DIR / "c_tuning_results.csv", index=False)
results_df.to_csv(REPORTS_DIR / "validation_results.csv", index=False)
results_df

## E11 - Raw Text Ablation

The final selected model uses raw tweet text. The vectorizer performs lowercasing internally so the raw and minimal-preprocessed variants can be compared directly.

In [ ]:
X_train_raw = train_df["tweet_text"]
X_val_raw = validation_df["tweet_text"]

e11_raw_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
        ),
        (
            "classifier",
            LogisticRegression(
                C=2.0,
                class_weight="balanced",
                solver="liblinear",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)
e11_raw_pipeline.fit(X_train_raw, y_train)
e11_predictions = e11_raw_pipeline.predict(X_val_raw)
e11_result = evaluate_predictions(
    "E11_raw_text_tfidf_1_2_lr_balanced_C2",
    y_val,
    e11_predictions,
    labels=class_names,
)
e11_result["name"] = "Raw text TF-IDF unigram/bigram + balanced Logistic Regression"
results_df = upsert_experiment(e11_result)
results_df

In [ ]:
e10_report_df = classification_report_dataframe(y_val, best_c_predictions, labels=class_names)
e11_report_df = classification_report_dataframe(y_val, e11_predictions, labels=class_names)

critical_recall_comparison_df = pd.DataFrame(
    [
        {
            "experiment": "E10_minimal",
            "missing_recall": e10_report_df.loc["missing_or_found_people", "recall"],
            "urgent_recall": e10_report_df.loc["requests_or_urgent_needs", "recall"],
        },
        {
            "experiment": "E11_raw",
            "missing_recall": e11_report_df.loc["missing_or_found_people", "recall"],
            "urgent_recall": e11_report_df.loc["requests_or_urgent_needs", "recall"],
        },
    ]
)
critical_recall_comparison_df

## E12-E16 - Controlled Preprocessing and TF-IDF Ablations

In [ ]:
def fit_raw_lr_ablation(
    experiment_id: str,
    *,
    min_df: int = 2,
    stop_words: str | None = None,
    sublinear_tf: bool = True,
    max_df: float = 1.0,
) -> tuple[Pipeline, dict, pd.DataFrame]:
    pipeline = Pipeline(
        [
            (
                "vectorizer",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=min_df,
                    max_df=max_df,
                    sublinear_tf=sublinear_tf,
                    lowercase=True,
                    stop_words=stop_words,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=2.0,
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    )
    pipeline.fit(X_train_raw, y_train)
    predictions = pipeline.predict(X_val_raw)
    result = evaluate_predictions(experiment_id, y_val, predictions, labels=class_names)
    report_df = classification_report_dataframe(y_val, predictions, labels=class_names)
    return pipeline, result, report_df


ablation_specs = [
    ("E12_raw_stopwords_tfidf_1_2_lr_balanced_C2", {"stop_words": "english"}),
    ("E13_raw_min_df5_tfidf_1_2_lr_balanced_C2", {"min_df": 5}),
    ("E14_raw_min_df1_tfidf_1_2_lr_balanced_C2", {"min_df": 1}),
    ("E15_raw_no_sublinear_tfidf_1_2_lr_balanced_C2", {"sublinear_tf": False}),
    ("E16_raw_max_df095_tfidf_1_2_lr_balanced_C2", {"max_df": 0.95}),
]

ablation_models: dict[str, Pipeline] = {}
ablation_reports: dict[str, pd.DataFrame] = {}

for experiment_id, kwargs in ablation_specs:
    pipeline, result, report_df = fit_raw_lr_ablation(experiment_id, **kwargs)
    ablation_models[experiment_id] = pipeline
    ablation_reports[experiment_id] = report_df
    results_df = upsert_experiment(result)

results_df.to_csv(REPORTS_DIR / "validation_results.csv", index=False)
results_df

In [ ]:
e13_result = next(result for result in experiment_results if result["experiment"] == "E13_raw_min_df5_tfidf_1_2_lr_balanced_C2")
e13_result

In [ ]:
e16_pipeline = ablation_models["E16_raw_max_df095_tfidf_1_2_lr_balanced_C2"]
e16_predictions = e16_pipeline.predict(X_val_raw)

e11_feature_count = len(e11_raw_pipeline.named_steps["vectorizer"].get_feature_names_out())
e16_feature_count = len(e16_pipeline.named_steps["vectorizer"].get_feature_names_out())
different_predictions = int((e11_predictions != e16_predictions).sum())

identical_feature_check = {
    "e11_feature_count": e11_feature_count,
    "e16_feature_count": e16_feature_count,
    "different_validation_predictions": different_predictions,
}
identical_feature_check

## Final Validation Selection and Train+Validation Retraining

E11 is selected on validation Macro-F1 with critical-class recall considered explicitly. The test set remains untouched until `03_evaluation_explainability.ipynb`.

In [ ]:
selected_experiment = "E11_raw_text_tfidf_1_2_lr_balanced_C2"
selected_validation_pipeline = e11_raw_pipeline
selected_validation_result = e11_result
selected_validation_report = e11_report_df

X_train_val_raw = pd.concat([X_train_raw, X_val_raw], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

final_pipeline = clone(selected_validation_pipeline)
final_pipeline.fit(X_train_val_raw, y_train_val)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

validation_model_path = MODELS_DIR / "selected_validation_model_e11.joblib"
final_model_path = MODELS_DIR / "final_e11_train_plus_validation.joblib"
joblib.dump(selected_validation_pipeline, validation_model_path)
joblib.dump(final_pipeline, final_model_path)

final_model_metadata = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "selected_experiment": selected_experiment,
    "selection_metric": "validation_macro_f1",
    "validation_metrics": {
        "accuracy": float(selected_validation_result["accuracy"]),
        "macro_f1": float(selected_validation_result["macro_f1"]),
        "weighted_f1": float(selected_validation_result["weighted_f1"]),
        "missing_recall": float(selected_validation_report.loc["missing_or_found_people", "recall"]),
        "urgent_recall": float(selected_validation_report.loc["requests_or_urgent_needs", "recall"]),
    },
    "training_data": {
        "train_samples": int(len(X_train_raw)),
        "validation_samples": int(len(X_val_raw)),
        "combined_samples": int(len(X_train_val_raw)),
    },
    "vectorizer_configuration": {
        "type": "TfidfVectorizer",
        "ngram_range": [1, 2],
        "min_df": 2,
        "sublinear_tf": True,
        "lowercase": True,
        "stop_words": None,
    },
    "classifier_configuration": {
        "type": "LogisticRegression",
        "C": 2.0,
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 1000,
        "random_state": 42,
    },
    "ablation_conclusion": {
        "raw_vs_minimal": "Raw text produced a marginally higher validation Macro-F1 with identical critical recalls.",
        "stopword_removal": "Reduced overall Macro-F1 and urgent recall.",
        "min_df_1": "Improved urgent recall slightly but reduced overall Macro-F1.",
        "min_df_5": "Reduced Macro-F1 without improving critical recalls.",
        "sublinear_tf": "Provided a small performance improvement.",
        "max_df_095": "Removed no features and changed no predictions.",
    },
    "test_set_status": "UNTOUCHED",
}

metadata_path = CHECKPOINTS_DIR / "final_model_selection.json"
metadata_path.write_text(json.dumps(final_model_metadata, indent=4, ensure_ascii=False), encoding="utf-8")
results_df.to_csv(REPORTS_DIR / "validation_results.csv", index=False)

print("Selected experiment:", selected_experiment)
print("Validation Macro-F1:", round(selected_validation_result["macro_f1"], 4))
print("Final train+validation samples:", len(X_train_val_raw))
print("Test set status: UNTOUCHED")